# Bayesian Unfolding

This notebook focuses on the traditional **Bayesian unfolding** of measured data (or smeared in our case).

Initially, we need to import ROOT with the RooUnfold library, and select data samples for training (creating the response matrix and unfolding matrix) and unfolding. We also create a folder for plots related to Bayesian unfolding.

In [ ]:
import os
import ROOT
ROOT.gSystem.Load("libRooUnfold") # load RooUnfold
ROOT.gStyle.SetOptStat(0) # disables the stat boxes in plots

# select data for training and unfolding by their seed
seedTraining = 2
seedUnfolding = 22

# create the folder
outDir = f"img/Bayes_{seedTraining}-{seedUnfolding}"
os.makedirs(outDir, exist_ok=True)

### 1D Distributions

We load the histograms and response matrices generated in the `plot-ea.ipynb` notebook. 

In [ ]:
# open files for reading
fileTraining = ROOT.TFile(f"data/{seedTraining}/events{seedTraining}_plots.root", "READ")
fileUnfolding = ROOT.TFile(f"data/{seedUnfolding}/events{seedUnfolding}_plots.root", "READ") # a different simulation -- represents the measured data

# create an empty dictionary where we will store our histograms and reponse matrices (RMs)
data = {} 

# define abbreviations, names, and latex labels of distributions we want to unfold
abbrevs = ['Nch', 'S0', 'S0pT1']
names = ['Multiplicity', 'Transverse Spherocity', 'Unweighted Transverse Spherocity']
latexs = ['N_{ch}^{}', 'S_{0}^{}', 'S_{0}^{p_{T} = 1}']

# fill the data dictionary
for abbrev, name, latex in zip(abbrevs, names, latexs):

    # syntax: (abbreviation, name, latex label): [true training data, smeared training data, response matrix from training data, independent true data, smeared data to be unfolded]
    data[(abbrev, name, latex)] = [fileTraining.Get(f"histo{seedTraining}{abbrev}{kind}").Clone(f"histo{seedTraining}{abbrev}{kind}_clone") for kind in ["True", "Smeared", "RM"]] # fill in the training data
    data[(abbrev, name, latex)] += [fileUnfolding.Get(f"histo{seedUnfolding}{abbrev}{kind}").Clone(f"histo{seedUnfolding}{abbrev}{kind}_clone") for kind in ["True", "Smeared"]] # append the data to be unfolded

    # memory management
    for histo in data[(abbrev, name, latex)]:
        histo.SetDirectory(0)

    # create folders for each observable
    os.makedirs(f"{outDir}/{abbrev}")

Afterwards, we construct response matrices compatible with RooUnfold for each observable from the training data. Then, we proceed with the Bayesian unfolding of the measured data and plot all distributions for comparison. Lastly, we perform a closure test to evaluate the quality of unfolding.

In [ ]:
# create a dictionary where unfolding matrices (UMs) will be stored and a dictionary to store unfolded distributions
UMs = {}
histosUnfolded = {}

# loop over different observables
for labels, histos in data.items():
    
    # define label and histogram variables from the data dictionary
    abbrev, name, latex = labels
    histoTrainingTrue, histoTrainingSmeared, histoTrainingRM, histoUnfoldingTrue, histoUnfoldingSmeared = histos

    # ---------
    # UNFOLDING
    # ---------

    # create a RooUnfold response matrix
    response = ROOT.RooUnfoldResponse(histoTrainingSmeared, histoTrainingTrue, histoTrainingRM)

    # Bayesian unfolding with nIter iterations
    nIter = 2 # define the number of iterations
    unfolded = ROOT.RooUnfoldBayes(response, histoTrainingSmeared) # initialize the unfolding object
    
    unfolded.SetIterations(nIter - 1) # perform nIter-1 iterations
    histoPenultimate = unfolded.Hunfold().Clone(f"histo{abbrev}Penultimate") # extract the histogram after the penultimate iteration (we must use the .Clone() method otherwise defining histoUnfolded below would overwrite this histogram)
    histoPenultimate.SetDirectory(0) # makes sure that Python takes memory management ownership of the object

    unfolded.SetIterations(nIter) # perform nIter iterations
    histoUnfolded = unfolded.Hunfold().Clone(f"histo{abbrev}Unfolded") # extract the unfolded histogram (we must once again use the .Clone() method for several reasons: .Hunfold() returns a pointer to an internal histogram managed by the RooUnfold object; if we modified (e.g., scaled) the histogram without cloning it, it would alter the internal state of the unfolding engine and if we then wanted to extract some more information about the RooUnfold object (e.g., a covariance matrix), the math would be broken)
    histoUnfolded.SetDirectory(0)

    UMs[abbrev] = unfolded.UnfoldingMatrix() # extract the unfolding matrix (UM)
    histosUnfolded[abbrev] = histoUnfolded.Clone(f"histo{abbrev}Unfolded") # extract the unfolded histogram
    histosUnfolded[abbrev].SetDirectory(0)

    # --------------
    # UNFOLDING PLOT
    # --------------

    # set up canvas for drawing
    canvasUnfolding = ROOT.TCanvas(f"canvas{abbrev}Unfolding", f"Bayesian Unfolding of {name}", 800, 600)
    canvasUnfolding.SetLogy() 

    # normalize the histograms
    for histo in [histoUnfoldingTrue, histoUnfoldingSmeared, histoPenultimate, histoUnfolded]:
        integral = histo.Integral()
        if integral > 0:
            histo.Scale(1.0 / integral)

    # customize the histograms (to customize title, labels, ..., we have to modify the first drawn histogram)
    histoUnfoldingTrue.SetTitle(f"Bayesian Unfolding of {name};{latex};Normalized Events") # set title and labels of axes

    maxVal = max(histoUnfoldingTrue.GetMaximum(), histoUnfoldingSmeared.GetMaximum(), histoPenultimate.GetMaximum(), histoUnfolded.GetMaximum())
    histoUnfoldingTrue.SetMaximum(maxVal * 7) # scale the y-axis to prevent cut-off

    histoUnfoldingTrue.SetLineColor(ROOT.kGreen)
    histoUnfoldingSmeared.SetLineColor(ROOT.kBlue)
    histoPenultimate.SetLineColor(ROOT.kOrange)
    histoUnfolded.SetLineColor(ROOT.kRed)

    histoPenultimate.SetMarkerStyle(ROOT.kOpenSquare)
    histoPenultimate.SetMarkerColor(ROOT.kOrange)
    histoPenultimate.SetMarkerSize(0.8)

    histoUnfolded.SetMarkerStyle(ROOT.kFullCircle)
    histoUnfolded.SetMarkerColor(ROOT.kRed)
    histoUnfolded.SetMarkerSize(0.5)

    # drawing
    histoUnfoldingTrue.Draw("HIST")
    histoUnfoldingSmeared.Draw("HIST SAME") # "SAME" tells ROOT to draw the histogram on top of the previous one
    histoPenultimate.Draw("PE SAME") # "P" forces a point scatter
    histoUnfolded.Draw("PE SAME") # "E" ensures that error bars are drawn

    # legend
    legend = ROOT.TLegend(0.69, 0.74, 0.89, 0.89) # set coordinates as fractions of the canvas (x1, y1, x2, y2)
    legend.SetBorderSize(0) # removes the border around the legend box
    
    legend.AddEntry(histoUnfoldingTrue, "True Data", "l")
    legend.AddEntry(histoUnfoldingSmeared, "Smeared Data", "l")
    legend.AddEntry(histoPenultimate, f"{nIter - 1}. Iteration", "p")
    legend.AddEntry(histoUnfolded, f"{nIter}. Iteration", "p")
    
    legend.Draw()

    # save
    canvasUnfolding.SaveAs(f"{outDir}/{abbrev}/{seedUnfolding}{abbrev}Unfolding.pdf") # save the canvas as a PDF file

    # ------------
    # CLOSURE TEST
    # ------------

    # define the histogram of the unfolded data and true data ratio
    histoRatio = histoUnfolded.Clone(f"closureRatio{abbrev}") # set the numerator = unfolded data of the ratio
    histoRatio.SetDirectory(0)
    histoRatio.Divide(histoUnfoldingTrue) # divide by the true data histogram

    # set up canvas
    canvasClosure = ROOT.TCanvas(f"canvas{abbrev}Closure", f"Closure Test for Unfolding of {name}", 800, 600)

    # customize the histogram
    histoRatio.SetTitle(f"Closure Test for Unfolding of {name};{latex};Unfolded / True Data")

    histoRatio.SetMinimum(0.7)
    histoRatio.SetMaximum(1.3)

    histoRatio.SetLineColor(ROOT.kPink)
    histoRatio.SetMarkerStyle(ROOT.kFullCircle)
    histoRatio.SetMarkerColor(ROOT.kPink)
    histoRatio.SetMarkerSize(0.5)

    # draw the histogram
    histoRatio.Draw("PE") 
    
    # draw the reference line at y=1
    xMin = histoRatio.GetXaxis().GetXmin() # extract x-axis limits
    xMax = histoRatio.GetXaxis().GetXmax()
    line = ROOT.TLine(xMin, 1.0, xMax, 1.0) # create a line between xMin, xMax at y=1 (TLine(x1, y1, x2, y2))
    line.SetLineColor(ROOT.kGreen)
    line.SetLineStyle(2) # dashed line
    line.SetLineWidth(2)
    line.Draw("SAME")

    # save
    canvasClosure.SaveAs(f"{outDir}/{abbrev}/{seedUnfolding}{abbrev}ClosureTest.pdf")

### $p_\mathrm{T}$ Spectra

We will carry out two different methods of unfolding 2D $p_\mathrm{T}$ spectra:
1. **no unfolding at all (method A):** In this method, we assume that it is irrelevant whether we use true or smeared observable distribution as their shapes are almost the same. Such assumption is valid mainly for spherocity observables and invalid for multiplicity, as the shapes differ substantially. Additionally, we are sorting the events into relative (spherocity, multiplicity, ...) classes, e.g., top $1 \: \%$ $S_0^{p_\mathrm{T} = 1}$ events, which is equal to the most jetty events. The method would not work if we used absolute cuts, e.g., events with $S_0^{p_\mathrm{T} = 1} > 0.8$, because the true and smeared observable distributions still vary, and we would therefore lose events which satisfy the absolute cut on particle-level but not on detector-level. We only sort the particles into classes, and then compute the value of the observable to obtain the absolute value of each cut.

2. **1D unfolding (method B):** For each $p_\mathrm{T}$ bin, we unfold the observable distribution (i.e., the $y$-axis in the 2D histogram) using the unfolding matrix acquired in the 1D unfolding above.

We are going to need NumPy to perform operations to find quantiles later.

In [ ]:
import numpy as np

Initially, we load the necessary 1D and 2D observable distributions from the files generated in the `plot-ea.ipynb` notebook, unfolded distributions, and unfolding matrices (UMs) from the previous step.

In [ ]:
# create an empty dictionary where we will store our histograms and unfolding matrices (RMs)
data2 = {} 

# fill the data dictionary
for abbrev, name, latex in zip(abbrevs, names, latexs):

    # syntax: (abbreviation, name, latex label): [training 2D histogram of the smeared observable vs true pT, training 1D histogram of the smeared observable, independent 2D histogram of the true observable vs pT, independent 1D histogram of the true observable, 1D histogram of the unfolded observable, unfolding matrix]
    data2[(abbrev, name, latex)] = [fileTraining.Get(f"histo{seedTraining}{kind}Smeared").Clone(f"histo{seedTraining}{kind}Smeared_clone2") for kind in [f"pT{abbrev}", abbrev]] # fill in the training data
    data2[(abbrev, name, latex)] += [fileUnfolding.Get(f"histo{seedUnfolding}{kind}True").Clone(f"histo{seedTraining}{kind}True_clone2") for kind in [f"pT{abbrev}", abbrev]] # append the data to be unfolded
    data2[(abbrev, name, latex)].append(histosUnfolded[abbrev]) # append the unfolded 1D distribution

    # memory management
    for histo in data2[(abbrev, name, latex)]:
        histo.SetDirectory(0)

    data2[(abbrev, name, latex)].append(UMs[abbrev]) # append the UM

We must ensure that the axis containing smeared data in the UMs are normalized in order that the values represent probability. Let's first check, whether the matrices are normalized in the $x$-axis.

In [ ]:
for abbrev, UM in UMs.items():
    nRows = UM.GetNrows()
    nCols = UM.GetNcols()

    # create a set to store column sums and a variable to determine whether UMs are normalized
    colSums = set()
    notNormalized = False

    # loop through each row
    for jCol in range(nCols):
        
        # sum across all columns in a given row
        colSum = sum(UM(iRow, jCol) for iRow in range(nRows))
        
        # store the sum in the set
        colSums.add(float(f"{colSum:.4f}"))
        
        # print the sum
        # print(f"{abbrev} Measured Bin {jCol} sum: {colSum:.4f}")
    
    if colSums != {0.0, 1.0}:
        notNormalized = True
        print(f"UM for {abbrev} is not normalized!")
    else:
        print(f"UM for {abbrev} is normalized.")

We can see that the UMs are normalized. If they are not, we have to normalize them by running the following cell.

In [ ]:
if notNormalized:
    
    # load the function for the x-axis normalization
    ROOT.gInterpreter.ProcessLine('#include "cpp/normalizeUM.cpp"')

    for abbrev, UM in UMs.items():

        # normalize the UM in the x-axis
        ROOT.normalizeUM(UM)

        nRows = UM.GetNrows()
        nCols = UM.GetNcols()

        # create a set to store column sums
        colSums = set()

        # loop through each row
        for jCol in range(nCols):
            
            # sum across all columns in a given row
            colSum = sum(UM(iRow, jCol) for iRow in range(nRows))
            
            # store the sum in the set
            colSums.add(float(f"{colSum:.4f}"))
            
            # print the sum
            # print(f"{abbrev} Measured Bin {jCol} sum: {colSum:.4f}")
        
        if colSums != {0.0, 1.0}:
            print(f"UM for {abbrev} is not normalized!")
        else:
            print(f"UM for {abbrev} is normalized.")

Now that we are sure that the UMs are properly normalized, we can proceed with the 1D unfolding. Since the observables are dependent on $p_\mathrm{T}$, we cannot just do a projection of the 2D histogram on the $y$-axis, as that would average out the $p_\mathrm{T}$ dependence. This is undesirable, because the observables are correlated with $p_\mathrm{T}$, e.g., highly spherical events will likely contain less high-$p_\mathrm{T}$ particles.

In [ ]:
# load the C++ applyUM(histoMeas, UM) function which applies the UM onto the pT bins in the 2D histograms of a smeared observable vs pT
ROOT.gInterpreter.ProcessLine('#include "cpp/applyUM.cpp"')

for labels, histos in data2.items():

    # define label variables, 2D histogram to be unfolded, and the UM from the data2 dictionary
    abbrev, name, latex = labels
    histopTObservableSmeared = histos[0]
    UM = histos[-1]

    # calculate the 2D histogram of the unfolded observable vs measured pT
    histopTObservableUnfolded = ROOT.applyUM(histopTObservableSmeared, UM)

    # store the histogram in the data2 dictionary
    data2[labels].insert(-2, histopTObservableUnfolded.Clone(f"histo{seedUnfolding}pT{abbrev}Unfolded_clone2"))
    data2[labels][-3].SetDirectory(0)
    # updated syntax: (abbreviation, name, latex label): [training 2D histogram of the smeared observable vs true pT, training 1D histogram of the smeared observable, independent 2D histogram of the true observable vs pT, independent 1D histogram of the true observable, 2D histogram of the unfolded observable vs pT, 1D histogram of the unfolded observable, unfolding matrix]

    # --------
    # PLOTTING -- Unfolded 2D Histogram
    # --------

    # set up canvas for drawing
    canvas2DUnfolded = ROOT.TCanvas(f"canvas{abbrev}2DUnfolded", f"Unfolded {name} vs " + "p_{T}", 800, 600)
    canvas2DUnfolded.SetLogz()

    # customize the histogram
    histopTObservableUnfolded.SetTitle(f"Unfolded {name} vs " + "p_{T};p_{T}^{MC} [GeV];" + latex[:-1] + " unf}")

    # draw and save
    histopTObservableUnfolded.Draw("COLZ")
    canvas2DUnfolded.SaveAs(f"{outDir}/{abbrev}/{seedTraining}pT{abbrev}Unfolded.pdf")

    # --------
    # PLOTTING -- Unfolding Matrix
    # --------

    # set up canvas for drawing
    canvasUM = ROOT.TCanvas(f"canvas{abbrev}UM", f"{name} Unfolding Matrix", 800, 600)
    canvasUM.SetLogz()

    # draw and save
    UM.Draw("COLZ")
    canvasUM.SaveAs(f"{outDir}/{abbrev}/{seedTraining}{abbrev}UM.pdf")

Then, we obtain the $p_\mathrm{T}$ spectra for various quantile classes:
1. multiplicity: top $1 \: \%$, $5 \: \%$, $10 \: \%$ (high-multiplicity events); 
2. spherocities: top $1 \: \%$, $5 \: \%$, $10 \: \%$ (spherical events), bottom $1 \: \%$, $5 \: \%$, $10 \: \%$ (jetty events). 

In [ ]:
for labels, histos in data2.items():
    
    # define label and histogram variables from the data dictionary
    abbrev, name, latex = labels
    histopTObservableSmeared, histoObservableSmeared, histopTObservableTrue, histoObservableTrue, histopTObservableUnfolded, histoObservableUnfolded, UM = histos

    # create folders for plots related to method A and B
    os.makedirs(f"{outDir}/{abbrev}/A", exist_ok=True)
    os.makedirs(f"{outDir}/{abbrev}/B", exist_ok=True)

    # define the quantile arrays with np.float64 data type to match ROOT's Double_t, which is required by the GetQuantiles() function
    quantilesBottom = np.array([0.01, 0.05, 0.1], dtype=np.float64)
    quantilesTop = np.array([0.9, 0.95, 0.99], dtype=np.float64)
    quantilesDict = {False: quantilesBottom, True: quantilesTop} # low quantiles are assigned a boolean False, high quantiles True

    # iterate through the quantile dictionary
    for kind, quantiles in quantilesDict.items():

        # skip low quantiles for multiplicity
        if abbrev == "Nch" and not kind:
            continue
        
        # define empty arrays for the output treshold values corresponding to the quantiles
        nQuantiles = len(quantiles)
        valuesTrue = np.zeros(nQuantiles, dtype=np.float64)
        valuesSmeared = np.zeros(nQuantiles, dtype=np.float64)
        valuesUnfolded = np.zeros(nQuantiles, dtype=np.float64)

        # calculate the treshold values for the given quantiles, we also calculate them for the unfolded 1D distributions to get the absolute value of our relative cuts for the legend
        histoObservableTrue.GetQuantiles(nQuantiles, valuesTrue, quantiles)
        histoObservableSmeared.GetQuantiles(nQuantiles, valuesSmeared, quantiles)
        histoObservableUnfolded.GetQuantiles(nQuantiles, valuesUnfolded, quantiles)

        # get the pT spectra for each quantile
        for i in range(nQuantiles):
            quantile = quantiles[i]

            # extract bin numbers for each quantile
            valueTrue = valuesTrue[i]
            valueSmeared = valuesSmeared[i]
            valueUnfolded = valuesUnfolded[i] # this is the absolute value of the relative cut

            # find the bin number in the 1D distribtion
            binTrue1D = histoObservableTrue.GetXaxis().FindBin(valueTrue)
            binSmeared1D = histoObservableSmeared.GetXaxis().FindBin(valueSmeared)
            binUnfolded1D = histoObservableUnfolded.GetXaxis().FindBin(valueUnfolded)

            # find the bin number in the 2D histograms
            binTrue2D = histopTObservableTrue.GetYaxis().FindBin(valueTrue)
            binSmeared2D = histopTObservableSmeared.GetYaxis().FindBin(valueSmeared)
            binUnfolded2D = histopTObservableUnfolded.GetYaxis().FindBin(valueUnfolded)

            # get the number of the last y-axis (observable) bin
            nBinsTrue2D = histopTObservableTrue.GetNbinsY()
            nBinsSmeared2D = histopTObservableSmeared.GetNbinsY()
            nBinsUnfolded2D = histopTObservableUnfolded.GetNbinsY()

            # calculate the x-axis projections 
            if kind:
                # this is executed for the top quantiles
                histopTTrue = histopTObservableTrue.ProjectionX(f"histo{seedUnfolding}pTTrueTop{100 - quantile*100}{abbrev}", binTrue2D, nBinsTrue2D + 1) # the +1 in nBinsTrue2D + 1 includes the overflow bin -- SHOULD I INCLUDE OVERFLOW (AND UNDERFLOW?)
                histopTTrue.SetDirectory(0)

                histopTSmeared = histopTObservableSmeared.ProjectionX(f"histo{seedUnfolding}pTSmearedTop{100 - quantile*100}{abbrev}", binSmeared2D, nBinsSmeared2D + 1)
                histopTSmeared.SetDirectory(0)

                histopTUnfolded = histopTObservableUnfolded.ProjectionX(f"histo{seedUnfolding}pTUnfoldedTop{100 - quantile*100}{abbrev}", binUnfolded2D, nBinsUnfolded2D + 1)
                histopTUnfolded.SetDirectory(0)

            else:
                # this is executed for the bottom quantiles
                histopTTrue = histopTObservableTrue.ProjectionX(f"histo{seedUnfolding}pTTrueBottom{quantile*100}{abbrev}", 0, binTrue2D) # include the underflow bin
                histopTTrue.SetDirectory(0)

                histopTSmeared = histopTObservableSmeared.ProjectionX(f"histo{seedUnfolding}pTSmearedBottom{quantile*100}{abbrev}", 0, binSmeared2D)
                histopTSmeared.SetDirectory(0)

                histopTUnfolded = histopTObservableUnfolded.ProjectionX(f"histo{seedUnfolding}pTUnfoldedBottom{quantile*100}{abbrev}", 0, binUnfolded2D)
                histopTUnfolded.SetDirectory(0)

            # ----------------------------------
            # PLOTTING -- METHOD A: NO UNFOLDING
            # ----------------------------------

            # set up canvas
            if kind:
                # this is executed for the top quantiles
                canvaspT = ROOT.TCanvas(f"canvasApTTop{100 - quantile}{abbrev}", "Method A: p_{T} spectrum for the top " + str(int(100 - quantile*100)) + f" % {latex} events", 800, 600)
            else:
                # this is executed for the bottom quantiles
                canvaspT = ROOT.TCanvas(f"canvasApTBottom{quantile}{abbrev}", "Method A: p_{T} spectrum for the bottom " + str(int(quantile*100)) + f" % {latex} events", 800, 600)
            canvaspT.SetLogy()

            # obtain per-event pT spectra by normalizing the pT histogram by the number of events
            histosObservable = [histoObservableTrue, histoObservableSmeared, histoObservableUnfolded]
            histospT = [histopTTrue, histopTSmeared, histopTUnfolded]
            histosBins1D = [binTrue1D, binSmeared1D, binUnfolded1D]

            for histoObservable, histopT, bin1D in zip(histosObservable, histospT, histosBins1D):
                nBinsObservable = histoObservable.GetNbinsX() # find the number of the last bin

                # we must distinguish between the low and high quantiles and calculate the number of events (the integral) in each quantile class
                if kind:
                    integral = histoObservable.Integral(bin1D, nBinsObservable + 1)
                else:
                    integral = histoObservable.Integral(0, bin1D)

                # normalize the pT spectrum
                if integral > 0:
                    histopT.Scale(1.0 / integral)

            # customize the histograms
            if kind:
                # this is executed for the top quantiles
                histopTTrue.SetTitle("Method A: p_{T} spectrum for the top " + str(int(100 - quantile*100)) + "% " + latex + " events;p_{T};Per-Event Yields")
            else:
                # this is executed for the bottom quantiles
                histopTTrue.SetTitle("Method A: p_{T} spectrum for the bottom " + str(int(quantile*100)) + "% " + latex + " events;p_{T};Per-Event Yields")

            histopTTrue.SetLineColor(ROOT.kGreen)
            histopTSmeared.SetLineColor(ROOT.kBlue)

            # drawing
            histopTTrue.Draw("HIST")
            histopTSmeared.Draw("HIST SAME")

            # legend
            legend = ROOT.TLegend(0.69, 0.74, 0.89, 0.89)
            legend.SetBorderSize(0)
            
            legend.AddEntry(histopTTrue, f"True {latex}", "l")
            legend.AddEntry(histopTSmeared, f"Smeared {latex}", "l")

            legend.Draw()

            # text with the absolute cut value
            text = ROOT.TLatex()
            text.SetNDC() # tell ROOT to use Normalized Device Coordinates (NDC), so that we can set the text position by fractions of the plot size
            text.SetTextFont(42)
            text.SetTextSize(0.03)
            text.SetTextAlign(21)

            if kind:
                # this is executed for the top quantiles
                text.DrawLatex(0.5, 0.83, f"unfolded {latex} > " + f"{valueUnfolded:.3f}")
            else:
                # this is executed for the bottom quantiles
                text.DrawLatex(0.5, 0.83, f"unfolded {latex} < " + f"{valueUnfolded:.3f}")

            # save
            if kind:
                # this is executed for the top quantiles
                canvaspT.SaveAs(f"{outDir}/{abbrev}/A/{seedUnfolding}ApTTop{100 - quantile*100}{abbrev}.pdf")
            else:
                # this is executed for the bottom quantiles
                canvaspT.SaveAs(f"{outDir}/{abbrev}/A/{seedUnfolding}ApTBottom{quantile*100}{abbrev}.pdf")

            # --------------------------------------
            # CLOSURE TEST -- METHOD A: NO UNFOLDING
            # --------------------------------------

            # define the histogram of the unfolded data and true data ratio
            if kind:
                # this is executed for the top quantiles
                histopTRatio = histopTSmeared.Clone(f"AclosureRatiopTTop{100 - quantile*100}{abbrev}") # set the numerator = unfolded data of the ratio
            else:
                # this is executed for the bottom quantiles
                histopTRatio = histopTSmeared.Clone(f"AclosureRatiopTBottom{quantile*100}{abbrev}") # set the numerator = unfolded data of the ratio
            histopTRatio.SetDirectory(0)
            histopTRatio.Divide(histopTTrue) # divide by the true data histogram

            # set up canvas
            if kind:
                # this is executed for the top quantiles
                canvaspTClosure = ROOT.TCanvas(f"canvasApTClosureTop{quantile}{abbrev}", "Method A: Closure Test for Unfolding of a p_{T} spectrum for the top " + str(int(100 - quantile*100)) + f"% {latex} events", 800, 600)
            else:
                # this is executed for the bottom quantiles
                canvaspTClosure = ROOT.TCanvas(f"canvasApTClosureTop{quantile}{abbrev}", "Method A: Closure Test for Unfolding of a p_{T} spectrum for the bottom " + str(int(quantile*100)) + f"% {latex} events", 800, 600)

            # customize the histogram
            if kind:
                # this is executed for the top quantiles
                histopTRatio.SetTitle("Method A: Closure Test for Unfolding of p_{T} spectrum for the top " + str(int(100 - quantile*100)) + f"% {latex} events;{latex};Unfolded / True Data")
            else:
                # this is executed for the bottom quantiles
                histopTRatio.SetTitle("Method A: Closure Test for Unfolding of p_{T} spectrum for the bottom " + str(int(quantile*100)) + f"% {latex} events;{latex};Unfolded / True Data")

            histopTRatio.SetMinimum(0.7)
            histopTRatio.SetMaximum(1.3)

            histopTRatio.SetLineColor(ROOT.kPink)
            histopTRatio.SetMarkerStyle(ROOT.kFullCircle)
            histopTRatio.SetMarkerColor(ROOT.kPink)
            histopTRatio.SetMarkerSize(0.5)

            # draw the histogram
            histopTRatio.Draw("PE") 
            
            # draw the reference line at y=1
            xMin = histopTRatio.GetXaxis().GetXmin() # extract x-axis limits
            xMax = histopTRatio.GetXaxis().GetXmax()
            line = ROOT.TLine(xMin, 1.0, xMax, 1.0) # create a line between xMin, xMax at y=1 (TLine(x1, y1, x2, y2))
            line.SetLineColor(ROOT.kGreen)
            line.SetLineStyle(2) # dashed line
            line.SetLineWidth(2)
            line.Draw("SAME")

            # save
            if kind:
                # this is executed for the top quantiles
                canvaspTClosure.SaveAs(f"{outDir}/{abbrev}/A/{seedUnfolding}ApTTop{100 - quantile*100}{abbrev}ClosureTest.pdf")
            else:
                # this is executed for the bottom quantiles
                canvaspTClosure.SaveAs(f"{outDir}/{abbrev}/A/{seedUnfolding}ApTBottom{quantile*100}{abbrev}ClosureTest.pdf")

            # ----------------------------------
            # PLOTTING -- METHOD B: 1D UNFOLDING
            # ----------------------------------

            # set up canvas
            if kind:
                # this is executed for the top quantiles
                canvaspT = ROOT.TCanvas(f"canvasBpTTop{100 - quantile*100}{abbrev}", "Method B: p_{T} spectrum for the top " + str(int(100 - quantile*100)) + f"% {latex} events", 800, 600)
            else:
                # this is executed for the bottom quantiles
                canvaspT = ROOT.TCanvas(f"canvasBpTBottom{quantile*100}{abbrev}", "Method B: p_{T} spectrum for the bottom " + str(int(quantile*100)) + f"% {latex} events", 800, 600)
            canvaspT.SetLogy()

            # customize the histograms
            if kind:
                # this is executed for the top quantiles
                histopTTrue.SetTitle("Method B: p_{T} spectrum for the top " + str(int(100 - quantile*100)) + "% " + latex + " events;p_{T};Per-Event Yields")
            else:
                # this is executed for the bottom quantiles
                histopTTrue.SetTitle("Method B: p_{T} spectrum for the bottom " + str(int(quantile*100)) + "% " + latex + " events;p_{T};Per-Event Yields")

            histopTTrue.SetLineColor(ROOT.kGreen)
            histopTSmeared.SetLineColor(ROOT.kBlue)
            histopTUnfolded.SetLineColor(ROOT.kRed)

            # drawing
            histopTTrue.Draw("HIST")
            histopTSmeared.Draw("HIST SAME")
            histopTUnfolded.Draw("HIST SAME")

            # legend
            legend = ROOT.TLegend(0.69, 0.74, 0.89, 0.89)
            legend.SetBorderSize(0)
            
            legend.AddEntry(histopTTrue, f"True {latex}", "l")
            legend.AddEntry(histopTSmeared, f"Smeared {latex}", "l")
            legend.AddEntry(histopTUnfolded, f"Unfolded {latex}", "l")
            
            legend.Draw()

            # text with the absolute cut value
            text = ROOT.TLatex()
            text.SetNDC() # tell ROOT to use Normalized Device Coordinates (NDC), so that we can set the text position by fractions of the plot size
            text.SetTextFont(42)
            text.SetTextSize(0.03)
            text.SetTextAlign(21)

            if kind:
                # this is executed for the top quantiles
                text.DrawLatex(0.5, 0.83, f"unfolded {latex} > " + f"{valueUnfolded:.3f}")
            else:
                # this is executed for the bottom quantiles
                text.DrawLatex(0.5, 0.83, f"unfolded {latex} < " + f"{valueUnfolded:.3f}")

            # save
            if kind:
                # this is executed for the top quantiles
                canvaspT.SaveAs(f"{outDir}/{abbrev}/B/{seedUnfolding}BpTTop{100 - quantile*100}{abbrev}.pdf")
            else:
                # this is executed for the bottom quantiles
                canvaspT.SaveAs(f"{outDir}/{abbrev}/B/{seedUnfolding}BpTBottom{quantile*100}{abbrev}.pdf")

            # --------------------------------------
            # CLOSURE TEST -- METHOD B: 1D UNFOLDING
            # --------------------------------------

            # define the histogram of the unfolded data and true data ratio
            if kind:
                # this is executed for the top quantiles
                histopTRatio = histopTUnfolded.Clone(f"BclosureRatiopTTop{100 - quantile*100}{abbrev}") # set the numerator = unfolded data of the ratio
            else:
                # this is executed for the bottom quantiles
                histopTRatio = histopTUnfolded.Clone(f"BclosureRatiopTBottom{quantile*100}{abbrev}") # set the numerator = unfolded data of the ratio
            histopTRatio.SetDirectory(0)
            histopTRatio.Divide(histopTTrue) # divide by the true data histogram

            # set up canvas
            if kind:
                # this is executed for the top quantiles
                canvaspTClosure = ROOT.TCanvas(f"canvasBpTClosureTop{100 - quantile*100}{abbrev}", "Method B: Closure Test for Unfolding of a p_{T} spectrum for the top " + str(int(100 - quantile*100)) + f"% {latex} events", 800, 600)
            else:
                # this is executed for the bottom quantiles
                canvaspTClosure = ROOT.TCanvas(f"canvasBpTClosureBottom{quantile*100}{abbrev}", "Method B: Closure Test for Unfolding of a p_{T} spectrum for the bottom " + str(int(quantile*100)) + f"% {latex} events", 800, 600)

            # customize the histogram
            if kind:
                # this is executed for the top quantiles
                histopTRatio.SetTitle("Method B: Closure Test for Unfolding of p_{T} spectrum for the top " + str(int(100 - quantile*100)) + f"% {latex} events;{latex};Unfolded / True Data")
            else:
                # this is executed for the bottom quantiles
                histopTRatio.SetTitle("Method B: Closure Test for Unfolding of p_{T} spectrum for the bottom " + str(int(quantile*100)) + f"% {latex} events;{latex};Unfolded / True Data")

            histopTRatio.SetMinimum(0.7)
            histopTRatio.SetMaximum(1.3)

            histopTRatio.SetLineColor(ROOT.kPink)
            histopTRatio.SetMarkerStyle(ROOT.kFullCircle)
            histopTRatio.SetMarkerColor(ROOT.kPink)
            histopTRatio.SetMarkerSize(0.5)

            # draw the histogram
            histopTRatio.Draw("PE") 
            
            # draw the reference line at y=1
            xMin = histopTRatio.GetXaxis().GetXmin() # extract x-axis limits
            xMax = histopTRatio.GetXaxis().GetXmax()
            line = ROOT.TLine(xMin, 1.0, xMax, 1.0) # create a line between xMin, xMax at y=1 (TLine(x1, y1, x2, y2))
            line.SetLineColor(ROOT.kGreen)
            line.SetLineStyle(2) # dashed line
            line.SetLineWidth(2)
            line.Draw("SAME")

            # save
            if kind:
                # this is executed for the top quantiles
                canvaspTClosure.SaveAs(f"{outDir}/{abbrev}/B/{seedUnfolding}BpTTop{100 - quantile*100}{abbrev}ClosureTest.pdf")
            else:
                # this is executed for the bottom quantiles
                canvaspTClosure.SaveAs(f"{outDir}/{abbrev}/B/{seedUnfolding}BpTBottom{quantile*100}{abbrev}ClosureTest.pdf")

In the end, we close the files we opened in the beginning.

In [ ]:
fileTraining.Close()
fileUnfolding.Close()